<a href="https://colab.research.google.com/github/kamathvk1982/GAI601/blob/main/AssignmentFour-EthicsAndExplainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 4: Ethics and Explainability**

Author(s): Vinayak Kamath

Course/Subject: GAI 601 Machine Learning

Date of Submission: May 10th, 2026

**Introduction**

In the context of machine learning and artificial intelligence (AI), ethics and explainability are increasingly becoming core considerations. As AI models are deployed in real-world applications, it is essential to ensure that they are not only accurate and efficient but also fair, transparent, and accountable. The complexity of modern machine learning models, such as deep neural networks and ensemble models, often makes them "black boxes" – decisions made by these models are not easily interpretable by humans. This lack of transparency can lead to ethical concerns, particularly in sensitive applications like healthcare, finance, and criminal justice.

Explainable AI (XAI) aims to address this issue by providing tools and techniques that make AI decisions more transparent and understandable. One such technique is Shapley values, which provide a mathematical way to explain the contribution of each feature to a model's prediction.

**Goal**

In this task, you will use SHAP to understand how your model makes predictions. You’ll find out which features are most important and think about whether your model is fair.

**Deliverables**

1.	SHAP Global Explainability
o	Load your trained model and dataset (Use a model you built in your previous assignment
o	Calculate SHAP values (using the SHAP explainer for your model e.g. TreeExplainer)
o	Create a SHAP summary bar plot to show which features have the biggest average impact on predictions
o	Identify the top 3 features and explain, in plain language, how they affect the model’s predictions

2.	SHAP Local Explanation
o	Pick one example from your data (a single item of data)
o	Use SHAP to show how each feature affected this single prediction
o	Describe which features influenced the prediction higher or lower

3.	Ethics and Fairness
o	Explain how SHAP can help make AI more ethical (e.g., transparency, bias detection, building trust).
o	Check if the model might rely too much on certain features (like income or location), which could reinforce bias.
o	Reflect on the earlier data science process (Assignments 2 & 3):
	What would you do differently to make the process more ethical and inclusive?
	What changes could reduce bias in future models?


## Building our Data Model and Tuning it

### 1.1. Data Loading

First, we'll load the `bank-full.csv` dataset. This dataset is semicolon-separated.

In [1]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/kamathvk1982/GAI601/main/bank-full.csv"
df = pd.read_csv(csv_url, sep=';')

print("Data loaded successfully. Displaying first 5 rows:")
display(df.head())

Data loaded successfully. Displaying first 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### 1.2. Data Preprocessing

For our model, we will predict the `y` column (whether the client subscribed to a term deposit). We will preprocess the data by dropping the `duration` column (to prevent target leakage) and encoding categorical features.

In [2]:
from sklearn.preprocessing import LabelEncoder

df_processed = df.copy()

# Drop 'duration' to avoid target leakage (if predicting before the call outcome)
columns_to_drop = ['duration']
df_processed = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns])

# Encode categorical features
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if 'y' in categorical_cols: # Ensure 'y' is not encoded as a feature yet
    categorical_cols.remove('y')

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])

# Define features (X) and target (y)
y = df_processed['y']
X = df_processed.drop(columns=['y'])

# Encode target variable 'y' ('yes'/'no' to 0/1)
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print("Processed data head:")
display(X.head())
print("Target variable (encoded) value counts:")
display(pd.Series(y_encoded).value_counts())

Processed data head:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome
0,58,4,1,2,0,2143,1,0,2,5,8,1,-1,0,3
1,44,9,2,1,0,29,1,0,2,5,8,1,-1,0,3
2,33,2,1,1,0,2,1,1,2,5,8,1,-1,0,3
3,47,1,1,3,0,1506,1,0,2,5,8,1,-1,0,3
4,33,11,2,3,0,1,0,0,2,5,8,1,-1,0,3


Target variable (encoded) value counts:


,count
0,39922
1,5289


We will split the preprocessed data into training and testing sets, then train the Decision Tree Classifier.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# Initialize and train the Decision Tree Classifier with default parameters
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print("Model training complete.")

Training set size: 31647 samples
Testing set size: 13564 samples
Model training complete.


### 1.3. Train a Model

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Calculate scale_pos_weight for imbalanced data
neg_count = pd.Series(y_train).value_counts()[0] # Assuming 0 is negative class
pos_count = pd.Series(y_train).value_counts()[1] # Assuming 1 is positive class
scale_pos_weight_value = neg_count / pos_count

# Define a more extensive parameter grid for LightGBM
param_grid_exp6 = {
    'n_estimators': [200, 300, 400, 500], # Expanding the range
    'learning_rate': [0.01, 0.02, 0.05],
    'num_leaves': [20, 31, 50, 70], # Wider range for leaves
    'max_depth': [7, 10, 12, 15], # Wider range for depth
    'min_child_samples': [20, 30, 40], # New hyperparameter to tune
    'subsample': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for bagging fraction
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # New hyperparameter for feature fraction
    'reg_alpha': [0, 0.1, 0.5], # L1 regularization
    'reg_lambda': [0, 0.1, 0.5], # L2 regularization
    'scale_pos_weight': [scale_pos_weight_value] # Keep the calculated value
}

# Initialize LightGBM Classifier with a random state and binary objective
lgbm_classifier_exp6 = lgb.LGBMClassifier(random_state=42, objective='binary', n_jobs=-1)

# Using RandomizedSearchCV for a wider search space to save time and computational resources
# We will use 100 iterations (n_iter) for a reasonable search
random_search_exp6 = RandomizedSearchCV(estimator=lgbm_classifier_exp6, param_distributions=param_grid_exp6,
                                        n_iter=100, cv=5, scoring='f1', verbose=1, random_state=42, n_jobs=-1)

# Fit RandomizedSearchCV to the training data
random_search_exp6.fit(X_train, y_train)

print("RandomizedSearchCV for Experiment 6 complete.")
print(f"Best parameters found for Experiment 6: {random_search_exp6.best_params_}")
print(f"Best cross-validation F1-score for Experiment 6: {random_search_exp6.best_score_:.4f}")

# Get the best model for Experiment 6
best_lgbm_model_exp6 = random_search_exp6.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits


### 1.4. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Make predictions with the best model from Experiment 6
y_pred_exp6 = best_lgbm_model_exp6.predict(X_test)

# Evaluate the tuned model from Experiment 6
accuracy_exp6 = accuracy_score(y_test, y_pred_exp6)
report_exp6 = classification_report(y_test, y_pred_exp6, target_names=le_target.classes_)
f1_exp6 = f1_score(y_test, y_pred_exp6, pos_label=1) # F1 for the 'yes' class

print(f"Accuracy (Experiment 6): {accuracy_exp6:.4f}")
print(f"F1-score for 'yes' class (Experiment 6): {f1_exp6:.4f}")
print("\nClassification Report (Experiment 6 Tuned Model):")
print(report_exp6)

# Compare with Experiment 5's metrics
print(f"\nAccuracy (Experiment 5): {accuracy_exp5:.4f}")
print(f"F1-score for 'yes' class (Experiment 5): {f1_exp5:.4f}")

if f1_exp6 > f1_exp5 and accuracy_exp6 >= accuracy_exp5 - 0.005: # Allow for slight accuracy drop for better F1
    print("\nHypothesis confirmed: Further LightGBM tuning improved F1-score and maintained/improved overall accuracy.")
elif f1_exp6 > f1_exp5:
    print("\nHypothesis partially confirmed: Further LightGBM tuning improved F1-score for the minority class, but with a trade-off in overall accuracy.")
else:
    print("\nHypothesis not strongly supported: Further LightGBM tuning did not significantly improve F1-score.")

print("\nWhat we learned: We pushed the LightGBM model to its limits with more extensive hyperparameter optimization, evaluating if more granular tuning could lead to further performance gains.")